<a href="https://colab.research.google.com/github/vladosG005/intro-ai/blob/main/lab1_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 1 Задача регрессии

## Задание
Решите пожалуйста следующие задачи.

1. Примените следующие типы регрессий для датасета представленного по
[ссылке](https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link).
- Линейная регрессия
- LASSO регрессия
- Ridge регрессия
- ElasticNet регрессия
- Polynom регрессия (для степени полинома 2)
- Polynom регрессия (для степени полинома 5)
- Polynom регрессия (для степени полинома 10)

Постройте модель для атрибутов `X = {SkinThickness, BMI}` и `Y = {Insulin}`.  
**Укажите `score` (r2_score) моделей. Найдите метрики MAE, MSE, RMSE.**


## Загрузка данных

Датасет доступен по ссылке <https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
! gdown --fuzzy 'https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link'

Downloading...
From: https://drive.google.com/uc?id=1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec
To: /content/diabetes.csv
100% 23.1k/23.1k [00:00<00:00, 39.7MB/s]


In [3]:
df = pd.read_csv('/content/diabetes.csv')
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


##  О датасете
[Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database/data)

Этот набор данных взят из Национального института диабета, заболеваний органов пищеварения и почек (NIDDK). Целью набора данных является диагностическое предсказание наличия у пациента диабета на основе определенных диагностических измерений, включенных в набор данных. На выбор этих экземпляров из более крупной базы данных было наложено несколько ограничений. В частности, все пациенты здесь — женщины не моложе 21 года, родом из индейцев Пима.

Наборы данных состоят из нескольких переменных медицинских предикторов и одной целевой переменной `Outcome`. Переменные-предикторы включают количество беременностей, которые были у пациентки, её индекс массы тела `BMI`, уровень инсулина, возраст и так далее. Всего в датасете 9 переменных:
- `Pregnancies` - количество беременностей;
- `Glucose` - концентрация глюкозы в плазме через 2 часа при пероральном тесте на толерантность к глюкозе;
- `BloodPressure` - диастолическое артериальное давление (мм рт. ст.);
- `SkinThickness` - толщина кожной складки трицепса (мм);
- `Insulin` - 2-часовой сывороточный инсулин (мкЕд/мл);
- `BMI` - Индекс массы тела (вес в кг/(рост в м)^2);
- `DiabetesPedigreeFunction` - функция оценки вероятности диабета на основе семейного анамнеза;
- `Age` - возраст (лет);
- `Outcome` - целевая переменная (0 или 1) 268 из 768 — 1 (больны), остальные — 0 (здоровы).



In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [6]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

Следует обратить внимание, что нулевые значения в столбцах `SkinThickness`, `BMI` и `Insulin` могут иметь содержательно некорректный характер: измерения толщины кожной складки, индекса массы тела и инсулина не должны интерпретироваться как физиологический ноль. Поэтому их рекомендуется рассматривать как скрытые пропуски.

## Краткие теоретические сведения

### Задача регрессии

Задача регрессии состоит в прогнозировании **числового значения** целевой переменной по набору признаков. В данной работе требуется предсказать уровень инсулина пациента по толщине кожной складки и индексу массы тела.

Общий вид линейной регрессии:

$$
\hat{y} = w_0 + w_1x_1 + w_2x_2
$$

где:

- $\hat{y}$ — прогнозируемое значение целевой переменной;
- $x_1$, $x_2$ — признаки;
- $w_0$ — свободный член;
- $w_1$, $w_2$ — коэффициенты модели.


### Регуляризация

Обычная линейная регрессия минимизирует сумму квадратов ошибок:

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

**Ridge-регрессия** добавляет L2-штраф на величину коэффициентов:

$$
MSE + \alpha \sum_{j=1}^{m} w_j^2
$$

Она уменьшает значения коэффициентов и повышает устойчивость модели при наличии взаимосвязанных признаков.

**LASSO-регрессия** использует L1-штраф:

$$
MSE + \alpha \sum_{j=1}^{m}|w_j|
$$

В отличие от Ridge, LASSO может занулять отдельные коэффициенты, то есть выполнять отбор признаков.

**ElasticNet** сочетает L1- и L2-регуляризацию. Это удобно, когда необходимо одновременно ограничить величины коэффициентов и сохранить возможность исключения слабых признаков.

### Полиномиальная регрессия

Полиномиальная регрессия расширяет исходные признаки степенями и произведениями. Например, для двух признаков при степени 2 формируются дополнительные признаки:

$$
x_1^2,\quad x_1x_2,\quad x_2^2
$$

Полиномиальная модель способна описывать нелинейные зависимости, но при чрезмерно высокой степени возрастает вероятность **переобучения**: модель хорошо описывает обучающую выборку, но плохо работает на новых данных.

### Метрики качества

| Метрика | Формула | Смысл | Направление улучшения |
| :-- | ------------------------------------ | :-- | :-- |
| $R^2$ | $$R^2 = 1 - \frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$$ | Доля вариации целевой переменной, объясняемая моделью | Ближе к 1 |
| MAE | $$MAE = \frac{1}{n}\sum(\|y_i-\hat{y}_i\|)$$ | Средняя абсолютная ошибка прогноза | Ближе к 0 |
| MSE | $$MSE = \frac{1}{n}\sum(y_i-\hat{y}_i)^2$$ | Средняя квадратичная ошибка | Ближе к 0 |
| RMSE | $$RMSE = \sqrt{MSE}$$ | Среднеквадратичная ошибка в единицах целевой переменной | Ближе к 0 |
| `score()` | | Для регрессоров `scikit-learn` обычно возвращает $R^2$ | Ближе к 1 |

Отрицательное значение коэффициента детерминации $R^2$ означает, что модель на тестовых данных работает хуже, чем простое предсказание среднего значения целевой переменной.


## Предобработка данных

In [7]:
X = df[['SkinThickness', 'BMI']]
y = df['Insulin']

X.shape, y.shape

((768, 2), (768,))

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=55)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((614, 2), (154, 2), (614,), (154,))

In [9]:
from sklearn.impute import SimpleImputer

zero_filler = SimpleImputer(missing_values=0, strategy="median")
X_train[['SkinThickness', 'BMI']] = zero_filler.fit_transform(X_train)
X_train.describe()

,SkinThickness,BMI
count,614.000000,614.000000
mean,28.975570,32.442671
std,8.931371,6.989666
min,7.000000,18.200000
25%,25.000000,27.425000
50%,29.000000,32.200000
75%,32.000000,36.575000
max,99.000000,67.100000


In [10]:
X_test[['SkinThickness', 'BMI']] = zero_filler.transform(X_test)

In [11]:
y_train.values[:] = zero_filler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_train.describe()

,Insulin
count,614.000000
mean,135.923453
std,84.835080
min,14.000000
25%,120.000000
50%,120.000000
75%,121.500000
max,846.000000


In [12]:
y_train.info()

<class 'pandas.core.series.Series'>
Index: 614 entries, 129 to 461
Series name: Insulin
Non-Null Count  Dtype
--------------  -----
614 non-null    int64
dtypes: int64(1)
memory usage: 9.6 KB


In [13]:
y_test.values[:] = zero_filler.transform(y_test.values.reshape(-1, 1)).flatten()

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train[['SkinThickness', 'BMI']] = scaler.fit_transform(X_train)
X_train.describe()

,SkinThickness,BMI
count,6.140000e+02,6.140000e+02
mean,-1.157236e-16,2.762900e-16
std,1.000815e+00,1.000815e+00
min,-2.462498e+00,-2.039337e+00
25%,-4.454872e-01,-7.184552e-01
50%,2.737529e-03,-3.474685e-02
75%,3.389061e-01,5.916875e-01
max,7.846670e+00,4.962409e+00


In [15]:
X_test[['SkinThickness', 'BMI']] = scaler.transform(X_test)

In [16]:
y_train.values[:] = scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_train.describe()

,Insulin
count,614.000000
mean,0.130293
std,0.813404
min,-1.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,8.000000


In [17]:
y_train.info()

<class 'pandas.core.series.Series'>
Index: 614 entries, 129 to 461
Series name: Insulin
Non-Null Count  Dtype
--------------  -----
614 non-null    int64
dtypes: int64(1)
memory usage: 9.6 KB


In [18]:
y_test.values[:] = scaler.transform(y_test.values.reshape(-1, 1)).flatten()

## Линейная регрессия

In [89]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.model_selection import KFold, cross_val_score

lin = LinearRegression()

kf = KFold(n_splits=5, shuffle=True, random_state=55)

cvs = cross_val_score(lin, X_train, y_train, cv=kf, scoring='r2')
print(cvs)
print(cvs.mean())

[-0.0367044  -0.05698955 -0.03777566  0.03785839 -0.41384075]
-0.1014903961888552


In [90]:
cvs = cross_val_score(lin, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
print(cvs)
print(cvs.mean())

[-0.44554166 -0.30638169 -0.34851633 -0.43124256 -0.27173032]
-0.36068251122460393


In [91]:
cvs = cross_val_score(lin, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.33142749 -0.32627016 -0.47262042 -0.98611724 -0.20821949]
-0.6649309606528495


In [92]:
cvs = cross_val_score(lin, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.15387499 -0.57120063 -0.68747394 -0.99303436 -0.45631074]
-0.7723789322969868


In [93]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

lin.fit(X_train, y_train)

y_pred = lin.predict(X_test)
r2_score(y_test, y_pred)

0.016116495457202618

In [95]:
mean_absolute_error(y_test, y_pred)

0.3768696720188272

In [96]:
mean_squared_error(y_test, y_pred)

0.8008469881807287

In [97]:
root_mean_squared_error(y_test, y_pred)

0.8949005465305788

## LASSO регрессия

In [98]:
lasso = Lasso(random_state=55)

cvs = cross_val_score(lasso, X_train, y_train, cv=kf, scoring='r2')
print(cvs)
print(cvs.mean())

[-0.05855848 -0.06587615 -0.00573337 -0.00359912 -0.13719268]
-0.05419195678505906


In [99]:
cvs = cross_val_score(lasso, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
print(cvs)
print(cvs.mean())

[-0.44799894 -0.31963969 -0.3371914  -0.43846141 -0.24390244]
-0.35743877601708807


In [100]:
cvs = cross_val_score(lasso, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.35949443 -0.32901326 -0.45802782 -1.02860783 -0.16747691]
-0.6685240504649326


In [101]:
cvs = cross_val_score(lasso, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.1659736  -0.57359677 -0.67677753 -1.01420305 -0.40923943]
-0.7679580761830701


In [104]:
lasso.fit(X_train, y_train)

y_pred = lasso.predict(X_test)
r2_score(y_test, y_pred)

-0.007378819216822707

In [105]:
mean_absolute_error(y_test, y_pred)

0.37594652904099163

In [68]:
mean_squared_error(y_test, y_pred)

0.8199713579929815

In [69]:
root_mean_squared_error(y_test, y_pred)

0.9055226987729139

## Ridge регрессия

In [106]:
ridge = Ridge(random_state=55)

cvs = cross_val_score(ridge, X_train, y_train, cv=kf, scoring='r2')
print(cvs)
print(cvs.mean())

[-0.03672432 -0.05685634 -0.03756294  0.03779561 -0.41324372]
-0.1013183402071075


In [107]:
cvs = cross_val_score(ridge, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
print(cvs)
print(cvs.mean())

[-0.44547548 -0.30631039 -0.34837514 -0.43125491 -0.27161365]
-0.36060591375576506


In [73]:
cvs = cross_val_score(ridge, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.33145307 -0.32622904 -0.47252354 -0.98618159 -0.20813157]
-0.6649037608727415


In [74]:
cvs = cross_val_score(ridge, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.15388607 -0.57116463 -0.68740348 -0.99306676 -0.45621439]
-0.7723470657532242


In [108]:
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_test)
r2_score(y_test, y_pred)

0.01614132266945867

In [76]:
mean_absolute_error(y_test, y_pred)

0.3768312545956253

In [77]:
mean_squared_error(y_test, y_pred)

0.8008267796925608

In [78]:
root_mean_squared_error(y_test, y_pred)

0.8948892555464955

## ElasticNet регрессия

In [109]:
elastic = ElasticNet(random_state=55)

cvs = cross_val_score(elastic, X_train, y_train, cv=kf, scoring='r2')
print(cvs)
print(cvs.mean())

[-0.05855848 -0.06587615 -0.00573337 -0.00359912 -0.13719268]
-0.05419195678505906


In [81]:
cvs = cross_val_score(elastic, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
print(cvs)
print(cvs.mean())

[-0.44799894 -0.31963969 -0.3371914  -0.43846141 -0.24390244]
-0.35743877601708807


In [82]:
cvs = cross_val_score(elastic, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.35949443 -0.32901326 -0.45802782 -1.02860783 -0.16747691]
-0.6685240504649326


In [83]:
cvs = cross_val_score(elastic, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.1659736  -0.57359677 -0.67677753 -1.01420305 -0.40923943]
-0.7679580761830701


In [157]:
elastic.fit(X_train, y_train)

y_pred = elastic.predict(X_test)
r2_score(y_test, y_pred)

0.0


-0.007378819216822707

In [85]:
mean_absolute_error(y_test, y_pred)

0.37594652904099163

In [86]:
mean_squared_error(y_test, y_pred)

0.8199713579929815

In [88]:
root_mean_squared_error(y_test, y_pred)

0.9055226987729139

## Polynom регрессия (для степени полинома 2)

In [112]:
from sklearn.preprocessing import PolynomialFeatures

polynomizer = PolynomialFeatures(degree=2)
poly2_X_train = pd.DataFrame(polynomizer.fit_transform(X_train))
poly2_X_train.describe()

,0,1,2,3,4,5
count,614.0,6.140000e+02,6.140000e+02,614.000000,614.000000,614.000000
mean,1.0,-1.157236e-16,2.762900e-16,1.000000,0.540683,1.000000
std,0.0,1.000815e+00,1.000815e+00,2.879955,1.209056,1.760395
min,1.0,-2.462498e+00,-2.039337e+00,0.000007,-1.034407,0.000037
25%,1.0,-4.454872e-01,-7.184552e-01,0.000007,-0.002192,0.086777
50%,1.0,2.737529e-03,-3.474685e-02,0.156658,0.012849,0.443304
75%,1.0,3.389061e-01,5.916875e-01,1.261801,0.720775,1.261023
max,1.0,7.846670e+00,4.962409e+00,61.570235,14.716382,24.625507


In [113]:
poly2_X_test = pd.DataFrame(polynomizer.transform(X_test))

In [114]:
poly2 = LinearRegression()

cvs = cross_val_score(poly2, poly2_X_train, y_train, cv=kf, scoring='r2')
print(cvs)
print(cvs.mean())

[-0.04328828 -0.07251386 -0.03107812 -0.15241095 -0.56047744]
-0.17195372999287395


In [115]:
cvs = cross_val_score(poly2, poly2_X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
print(cvs)
print(cvs.mean())

[-0.45858622 -0.30934177 -0.35491473 -0.47553031 -0.30334509]
-0.38034362412248834


In [116]:
cvs = cross_val_score(poly2, poly2_X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.33988309 -0.33106218 -0.46957025 -1.18112791 -0.22981501]
-0.7102916875252265


In [117]:
cvs = cross_val_score(poly2, poly2_X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.15753319 -0.57538003 -0.68525196 -1.08679709 -0.47939024]
-0.7968705029676804


In [155]:
poly2.fit(poly2_X_train, y_train)
y_pred = poly2.predict(poly2_X_train)
y_pred = poly2.predict(poly2_X_test)
r2_score(y_test, y_pred)

0.026027608879366726


0.013482106819751039

In [143]:
mean_absolute_error(y_test, y_pred)

0.3917477933591236

In [137]:
mean_squared_error(y_test, y_pred)

0.8029912890011607

In [138]:
root_mean_squared_error(y_test, y_pred)

0.8960978121841168

## Polynom регрессия (для степени полинома 5)

In [128]:
polynomizer = PolynomialFeatures(degree=5)
poly5_X_train = pd.DataFrame(polynomizer.fit_transform(X_train))
poly5_X_train.describe()

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
count,614.0,6.140000e+02,6.140000e+02,614.000000,614.000000,614.000000,6.140000e+02,6.140000e+02,614.000000,614.000000,...,6.140000e+02,6.140000e+02,614.000000,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,614.000000,6.140000e+02
mean,1.0,-1.157236e-16,2.762900e-16,1.000000,0.540683,1.000000,9.876591e-01,2.431032e-01,0.313870,0.701601,...,2.180279e+00,1.751775e+00,1.963931,4.093944e+00,5.144321e+01,4.541661e+00,2.644993e+00,3.093827e+00,4.675179,1.012870e+01
std,0.0,1.000815e+00,1.000815e+00,2.879955,1.209056,1.760395,1.995597e+01,3.413829e+00,3.647962,6.638594,...,1.216915e+01,1.052276e+01,13.775351,2.792085e+01,1.201079e+03,6.193445e+01,3.666553e+01,4.008482e+01,59.412304,1.302861e+02
min,1.0,-2.462498e+00,-2.039337e+00,0.000007,-1.034407,0.000037,-1.493234e+01,-7.577065e+00,-7.025004,-8.481386,...,-2.407447e+00,2.797553e-10,-3.560554,1.393547e-09,-9.054821e+01,-3.471063e+01,-2.850254e+01,-2.642586e+01,-24.500484,-3.527319e+01
25%,1.0,-4.454872e-01,-7.184552e-01,0.000007,-0.002192,0.086777,-8.841088e-02,-5.126978e-02,-0.066718,-0.370933,...,-1.679549e-08,1.018700e-05,-0.000424,7.530174e-03,-1.754592e-02,-1.240708e-02,-1.220413e-02,-1.004943e-02,-0.012463,-1.915661e-01
50%,1.0,2.737529e-03,-3.474685e-02,0.156658,0.012849,0.443304,2.051522e-08,-4.750026e-07,0.001087,-0.000042,...,3.048228e-04,2.235719e-02,0.002378,1.965207e-01,1.537424e-13,-3.559700e-12,8.872649e-09,-1.908329e-09,0.000087,-5.064971e-08
75%,1.0,3.389061e-01,5.916875e-01,1.261801,0.720775,1.261023,3.892585e-02,6.299581e-02,0.039307,0.207214,...,8.918794e-01,5.972739e-01,0.411500,1.590179e+00,4.470919e-03,9.163276e-03,5.612814e-03,7.028754e-03,0.015184,7.259937e-02
max,1.0,7.846670e+00,4.962409e+00,61.570235,14.716382,24.625507,4.831213e+02,5.610838e+01,56.803541,122.201849,...,2.139215e+02,2.165719e+02,233.123567,6.064156e+02,2.974589e+04,1.225276e+03,8.257123e+02,8.359426e+02,1156.854587,3.009283e+03


In [129]:
poly5_X_test = pd.DataFrame(polynomizer.transform(X_test))

In [130]:
poly5 = LinearRegression()

cvs = cross_val_score(poly5, poly5_X_train, y_train, cv=kf, scoring='r2')
print(cvs)
print(cvs.mean())

[-2.73476213e-02 -1.22188348e+00 -8.92913695e+00 -1.41893619e+04
 -1.69766831e+00]
-2840.2475971612316


In [131]:
cvs = cross_val_score(poly5, poly5_X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
print(cvs)
print(cvs.mean())

[ -0.45693745  -0.37412762  -0.57780992 -11.32876508  -0.37858052]
-2.6232441202400336


In [132]:
cvs = cross_val_score(poly5, poly5_X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
print(cvs)
print(cvs.mean())

[-1.31941069e+00 -6.85848097e-01 -4.52189527e+00 -1.45439720e+04
 -3.97291652e-01]
-2910.1792891418363


In [133]:
cvs = cross_val_score(poly5, poly5_X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
print(cvs)
print(cvs.mean())

[  -1.14865604   -0.82815946   -2.12647485 -120.59839137   -0.63031076]
-25.06639849377567


In [144]:
poly5.fit(poly5_X_train, y_train)

y_pred = poly5.predict(poly5_X_test)
r2_score(y_test, y_pred)

-0.014660075952506402

In [145]:
mean_absolute_error(y_test, y_pred)

0.4104151323313896

In [146]:
mean_squared_error(y_test, y_pred)

0.8258980479923758

In [147]:
root_mean_squared_error(y_test, y_pred)

0.9087893309190946

## Polynom регрессия (для степени полинома 10)

In [45]:
polynomizer = PolynomialFeatures(degree=10)
poly10_X_train = pd.DataFrame(polynomizer.fit_transform(X_train))
poly10_X_train.describe()

,0,1,2,3,4,5,6,7,8,9,...,56,57,58,59,60,61,62,63,64,65
count,614.0,6.140000e+02,6.140000e+02,614.000000,614.000000,614.000000,6.140000e+02,6.140000e+02,614.000000,614.000000,...,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02
mean,1.0,-1.157236e-16,2.762900e-16,1.000000,0.540683,1.000000,9.876591e-01,2.431032e-01,0.313870,0.701601,...,6.089543e+04,3.850255e+03,1.448554e+03,1.349167e+03,1.410862e+03,1.613748e+03,2.150521e+03,3.545930e+03,7.196008e+03,1.704941e+04
std,0.0,1.000815e+00,1.000815e+00,2.879955,1.209056,1.760395,1.995597e+01,3.413829e+00,3.647962,6.638594,...,1.471071e+06,6.636122e+04,2.748004e+04,2.766891e+04,2.812581e+04,2.938823e+04,3.534617e+04,6.125848e+04,1.435029e+05,3.668502e+05
min,1.0,-2.462498e+00,-2.039337e+00,0.000007,-1.034407,0.000037,-1.493234e+01,-7.577065e+00,-7.025004,-8.481386,...,-1.607932e+02,1.177418e-25,-3.439008e+00,5.865080e-25,-1.184291e+00,2.921577e-24,-1.311375e+01,1.455328e-23,-1.452097e+02,7.249436e-23
25%,1.0,-4.454872e-01,-7.184552e-01,0.000007,-0.002192,0.086777,-8.841088e-02,-5.126978e-02,-0.066718,-0.370933,...,-7.068790e-24,4.664429e-21,-6.323179e-19,9.208577e-16,-5.074570e-14,9.095120e-11,-1.135290e-09,2.147479e-07,-4.259692e-07,4.920537e-06
50%,1.0,2.737529e-03,-3.474685e-02,0.156658,0.012849,0.443304,2.051522e-08,-4.750026e-07,0.001087,-0.000042,...,6.448041e-10,1.041655e-04,1.195335e-09,7.360751e-05,3.831495e-10,7.038236e-05,6.809430e-08,2.021539e-04,3.458482e-06,1.712195e-02
75%,1.0,3.389061e-01,5.916875e-01,1.261801,0.720775,1.261023,3.892585e-02,6.299581e-02,0.039307,0.207214,...,1.846433e+00,1.078859e+00,5.122100e-01,3.703911e-01,1.948606e-01,1.759999e-01,1.285665e-01,1.979546e-01,2.198157e-01,3.188710e+00
max,1.0,7.846670e+00,4.962409e+00,61.570235,14.716382,24.625507,4.831213e+02,5.610838e+01,56.803541,122.201849,...,3.644694e+07,1.501302e+06,6.734570e+05,6.818009e+05,6.902481e+05,6.988000e+05,7.074579e+05,1.338313e+06,3.481302e+06,9.055781e+06


In [46]:
poly10_X_test = pd.DataFrame(polynomizer.transform(X_test))

In [148]:
poly10 = LinearRegression()

cvs = cross_val_score(poly10, poly10_X_train, y_train, cv=kf, scoring='r2')
print(cvs)
print(cvs.mean())

[-3.71686508e-01 -7.66909436e+05 -5.25298583e+07 -5.96850356e+12
 -1.35840714e+02]
-1193711371774.3032


In [149]:
poly10.fit(poly10_X_train, y_train)

y_pred = poly10.predict(poly10_X_test)
r2_score(y_test, y_pred)

-8.803554061775648

In [150]:
mean_absolute_error(y_test, y_pred)

0.6639057056699814

In [151]:
mean_squared_error(y_test, y_pred)

7.979752386933594

In [152]:
root_mean_squared_error(y_test, y_pred)

2.8248455509874506

## Вывод

Сделайте вывод о применимости разных видов регрессии для решения этой задачи.

Разобранные здесь виды регрессии практически не применимы к данной задаче, так как показывают результаты, очень близкие к угадыванию. Полиномиальная регрессия с ростом степени показывает всё более худщие результаты.